In [1]:
from google.colab import drive 
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/MorDIFF

/content/drive/MyDrive/MorDIFF


In [3]:
# Define datasets we are going to be working with, for readablity
DATASET_PATH = "/content/drive/MyDrive/Datasets/VGGFace2_filtered_10_first300"


In [4]:
DATASET_NAME = DATASET_PATH.split('/')[-1]
print(f"Working with dataset: {DATASET_NAME}")
ALIGNED_PATH = "/content/drive/MyDrive/Datasets_aligned"

Working with dataset: VGGFace2_filtered_10_first300


In [5]:
import importlib
import subprocess
import sys

# Install OS-level dependencies needed by dlib and image processing.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    'cmake', 'build-essential', 'libgl1', 'libglib2.0-0'
], check=True)

# Use the exact interpreter backing this notebook kernel. Colab can expose
# separate /usr and /usr/local Python installations to shell commands.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--system', '--python', sys.executable, '--no-deps',
    'dlib-bin',
    'pytorch-lightning==1.9.5',
    'lightning-utilities==0.11.9',
    'torchmetrics==0.11.4',
    'PyYAML',
    'packaging',
    'fsspec',
    'pytorch-fid==0.3.0',
    'lpips==0.1.4',
    'lmdb',
    'facenet-pytorch==2.5.3'
])

mods = [
    'torch', 'torchvision', 'PIL', 'numpy', 'scipy', 'requests', 'tqdm',
    'matplotlib', 'skimage', 'cv2', 'dlib', 'facenet_pytorch', 'lpips', 'lmdb'
]
missing = []
for module in mods:
    try:
        importlib.import_module(module)
    except Exception as error:
        missing.append(f'{module}: {type(error).__name__}: {error}')
if missing:
    raise RuntimeError(f'Missing required dependencies: {missing}')

print(f'All notebook dependencies are available in {sys.executable}.')
print(f'NumPy: {__import__("numpy").__version__}')
print(f'PyTorch: {__import__("torch").__version__}')
print(f'TorchVision: {__import__("torchvision").__version__}')

All notebook dependencies are available in /usr/bin/python3.
NumPy: 2.0.2
PyTorch: 2.11.0+cu128
TorchVision: 0.26.0+cu128


In [6]:
# align dataset images while preserving the original structure
import os
import glob
import bz2
import requests
import numpy as np
import scipy.ndimage
import PIL.Image
import dlib
from tqdm import tqdm

# --- 1. Automated Model Weight Downloader ---
def get_shape_predictor():
    url = 'http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2'
    bz2_path = 'temp/shape_predictor_68_face_landmarks.dat.bz2'
    dat_path = 'temp/shape_predictor_68_face_landmarks.dat'
    
    if os.path.exists(dat_path):
        return dat_path
        
    os.makedirs('temp', exist_ok=True)
    if not os.path.exists(bz2_path):
        print("Downloading dlib 68-face landmarks model...")
        response = requests.get(url, stream=True)
        with open(bz2_path, 'wb') as f:
            f.write(response.content)
            
    print("Decompressing landmarks model...")
    with open(dat_path, 'wb') as f_out, bz2.BZ2File(bz2_path, 'rb') as f_in:
        f_out.write(f_in.read())
        
    return dat_path

# --- 2. FFHQ Standard Image Alignment Transform ---
def image_align(src_file, dst_file, face_landmarks, output_size=256, transform_size=4096, enable_padding=True):
    lm = np.array(face_landmarks)
    lm_eye_left = lm[36:42]
    lm_eye_right = lm[42:48]
    lm_mouth_outer = lm[48:60]

    eye_left = np.mean(lm_eye_left, axis=0)
    eye_right = np.mean(lm_eye_right, axis=0)
    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_left = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2

    img = PIL.Image.open(src_file).convert('RGB')

    shrink = int(np.floor(qsize / output_size * 0.5))
    if shrink > 1:
        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))
        img = img.resize(rsize, PIL.Image.LANCZOS)
        quad /= shrink
        qsize /= shrink

    border = max(int(np.rint(qsize * 0.1)), 3)
    crop = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
            int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0),
            min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))
    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:
        img = img.crop(crop)
        quad -= crop[0:2]

    pad = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
           int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0),
           max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))
    if enable_padding and max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        img = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect')
        h, w, _ = img.shape
        y_grid, x_grid, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(
            1.0 - np.minimum(np.float32(x_grid) / pad[0], np.float32(w - 1 - x_grid) / pad[2]),
            1.0 - np.minimum(np.float32(y_grid) / pad[1], np.float32(h - 1 - y_grid) / pad[3])
        )
        blur = qsize * 0.02
        img += (scipy.ndimage.gaussian_filter(img, [blur, blur, 0]) - img) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img += (np.median(img, axis=(0, 1)) - img) * np.clip(mask, 0.0, 1.0)
        img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')
        quad += pad[:2]

    img = img.transform((transform_size, transform_size), PIL.Image.QUAD, (quad + 0.5).flatten(), PIL.Image.BILINEAR)
    if output_size < transform_size:
        img = img.resize((output_size, output_size), PIL.Image.LANCZOS)

    img.save(dst_file, 'PNG')

# --- 3. Recursive Dataset Pipeline ---
def align_dataset(input_root, output_root):
    model_path = get_shape_predictor()
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(model_path)

    # Recursively gather images across all subfolders
    extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(input_root, '**', ext), recursive=True))

    print(f"Discovered {len(image_paths)} images across dataset tree.")

    for img_path in tqdm(image_paths, desc="Aligning Faces"):
        rel_path = os.path.relpath(img_path, input_root)
        name_without_ext = os.path.splitext(rel_path)[0]
        out_path = os.path.join(output_root, f"{name_without_ext}.png")

        if os.path.exists(out_path):
            continue

        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        try:
            img = dlib.load_rgb_image(img_path)
            dets = detector(img, 1)
            if len(dets) == 0:
                continue

            # Align using primary detected face
            landmarks = [(p.x, p.y) for p in predictor(img, dets[0]).parts()]
            image_align(img_path, out_path, landmarks, output_size=256)
        except Exception as e:
            print(f"Skipping {img_path}: {e}")

    print(f"Alignment completed. Files saved to: {output_root}")

# --- Execution ---
align_dataset(
    input_root=DATASET_PATH, 
    output_root=f"{ALIGNED_PATH}/{DATASET_NAME}"
)

Discovered 900 images across dataset tree.


Aligning Faces:   0%|          | 1/900 [00:00<07:45,  1.93it/s]/tmp/ipykernel_339/3378023141.py:92: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')
Aligning Faces: 100%|██████████| 900/900 [20:49<00:00,  1.39s/it]

Alignment completed. Files saved to: /content/drive/MyDrive/Datasets_aligned/VGGFace2_filtered_10_first300


In [7]:
import os
import torch
import numpy as np
import pandas as pd
from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

guideline_dir = f"/content/drive/MyDrive/Morph_guidelines/{DATASET_NAME}"
guideline_path = os.path.join(guideline_dir, "morph_guidelines.csv")

# do the guideline generation only if it doesn't exist already
if not os.path.exists(guideline_path):
    os.makedirs(guideline_dir, exist_ok=True)

    # get GPU if available, else fallback to CPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    # get the list of all aligned images
    aligned_dir = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
    image_files = sorted([
        os.path.join(root, f) 
        for root, _, files in os.walk(aligned_dir) 
        for f in files if f.lower().endswith(('.png', '.jpg'))
    ])

    embeddings = []
    valid_files = []
    subject_ids = []

    # 1. Extract embeddings and parse subject IDs
    with torch.no_grad():
        for f in tqdm(image_files, desc="Extracting embeddings"):
            # Extracts ID from parent folder (e.g., '00491') or fallback to filename prefix
            subject_id = os.path.basename(os.path.dirname(f))
            if not subject_id or subject_id == os.path.basename(aligned_dir):
                subject_id = os.path.basename(f).split('_')[0]

            # Compute embedding
            img = Image.open(f).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            emb = resnet(tensor).cpu().numpy().flatten()
            norm = np.linalg.norm(emb)
            if norm > 0:
                emb = emb / norm
                embeddings.append(emb)
                valid_files.append(f)
                subject_ids.append(subject_id)

    embeddings = np.array(embeddings)
    subject_ids = np.array(subject_ids)

    # 2. Compute Cosine Similarity Matrix
    similarity_matrix = np.dot(embeddings, embeddings.T)

    # 3. Mask out all images belonging to the same identity
    # Creates a boolean matrix where True indicates same subject
    same_identity_mask = subject_ids[:, None] == subject_ids[None, :]
    # Invalidate self and same-subject matches
    # basically set super low similarity score to the images beloginging to same identity
    similarity_matrix[same_identity_mask] = -1.0  

    # 4. Pair distinct identities greedily based on highest similarity
    morph_pairs = []
    # tracking used identites instead of single probes to get best, but one morph per subject
    used_subject_ids = set()

    for i in range(len(valid_files)):
        # skip if this subject already has morph suggested
        current_id = subject_ids[i]
        if current_id in used_subject_ids:
            continue

        # copy the scores for the current subject to avoid unnecesary iterations
        scores = similarity_matrix[i].copy()

        # Mask out all images belonging to already-paired identities
        for used_id in used_subject_ids:
            scores[subject_ids == used_id] = -1.0

        best_match_idx = int(np.argmax(scores))
        sim_score = scores[best_match_idx]
        best_id = subject_ids[best_match_idx]

        # Ensure valid inter-subject match meets threshold
        if sim_score >= 0.40 and best_id not in used_subject_ids:
                morph_pairs.append({
                    "id1": current_id,
                    "id2": best_id,
                    "img1": valid_files[i],
                    "img2": valid_files[best_match_idx],
                    "similarity": float(sim_score)
                })
                used_subject_ids.add(current_id)
                used_subject_ids.add(best_id)

    # 5. Save to CSV
    df_pairs = pd.DataFrame(morph_pairs)
    df_pairs.to_csv(guideline_path, index=False)
    print(f"Generated {len(df_pairs)} inter-subject morphing pairs -> {guideline_path}")

else:
    print(f"Morph guidelines already exist at {guideline_path}. Skipping generation.")

  0%|          | 0.00/107M [00:00<?, ?B/s]

Extracting embeddings: 100%|██████████| 854/854 [00:26<00:00, 32.61it/s]


Generated 142 inter-subject morphing pairs -> /content/drive/MyDrive/Morph_guidelines/VGGFace2_filtered_10_first300/morph_guidelines.csv


In [8]:
import os
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from torch.nn import functional as F
from tqdm import tqdm

# ----------------------------
# setup
# ----------------------------
# dataset name is defined at the top already
aligned_dir = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
suggestions_csv = f"/content/drive/MyDrive/Morph_guidelines/{DATASET_NAME}/morph_guidelines.csv"
output_root = f"/content/drive/MyDrive/Morphs/mor_diff/{DATASET_NAME}"
os.makedirs(output_root, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# load model once
# ----------------------------
path_to_diff_model = "./diffae-master"
sys.path.append(path_to_diff_model)
from templates import *

conf = ffhq256_autoenc()
model = LitModel(conf)
state = torch.load(
    f"{path_to_diff_model}/checkpoints/{conf.name}/last.ckpt",
    map_location="cpu",
    weights_only=False
)
model.load_state_dict(state["state_dict"], strict=False)
model.ema_model.eval()
model.ema_model.to(device)

image_size = conf.img_size
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
to_pil = transforms.ToPILImage()

def load_aligned(path):
    img = Image.open(path).convert("RGB")
    if img.size[0] != image_size or img.size[1] != image_size:
        img = img.resize((image_size, image_size), Image.LANCZOS)
    return transform(img)

def cos_sim(a, b):
    a = F.normalize(a.view(-1), dim=0)
    b = F.normalize(b.view(-1), dim=0)
    return (a * b).sum()

# ----------------------------
# read suggestions csv
# ----------------------------
df = pd.read_csv(suggestions_csv)

# using only images columns for morph generation
pairs = list(zip(df["img1"], df["img2"]))

# ----------------------------
# morph generation
# ----------------------------
alpha = torch.tensor([0.0, 0.5, 1.0], device=device)

with torch.no_grad():
    for img1_path, img2_path in tqdm(pairs, desc="Generating morphs"):
        # preserve names from the source images
        name1 = os.path.splitext(os.path.basename(img1_path))[0]
        name2 = os.path.splitext(os.path.basename(img2_path))[0]

        final_name = f"morphed_{name1}_and_{name2}.png"
        output_path = os.path.join(output_root, final_name)

        if os.path.exists(output_path):
            continue

        img1 = load_aligned(img1_path).unsqueeze(0).to(device)
        img2 = load_aligned(img2_path).unsqueeze(0).to(device)
        batch = torch.cat([img1, img2], dim=0)

        cond = model.encode(batch)
        xT = model.encode_stochastic(batch, cond, T=250)

        intp = cond[0][None] * (1 - alpha[:, None]) + cond[1][None] * alpha[:, None]
        theta = torch.arccos(cos_sim(xT[0], xT[1]))
        x_shape = xT[0].shape

        intp_x = (
            torch.sin((1 - alpha[:, None]) * theta) * xT[0].flatten(0, 2)[None] +
            torch.sin(alpha[:, None] * theta) * xT[1].flatten(0, 2)[None]
        ) / torch.sin(theta)
        intp_x = intp_x.view(-1, *x_shape)

        pred = model.render(intp_x, intp, T=20)

        # save only the middle morph, which is the meaningful blend
        morph_img = to_pil(pred[1].cpu())
        morph_img.save(output_path)

INFO:lightning_fabric.utilities.seed:Global seed set to 0


Model params: 160.69 M


Generating morphs:   0%|          | 0/142 [00:00<?, ?it/s]

/content/drive/MyDrive/MorDIFF/diffae-master/diffusion/base.py:306: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.conf.fp16):
